# FinSolve Technologies — RAG Evaluation

This notebook documents how the RAG system was tested and improved. It moves from retrieval checks to answer generation, conversation, access control, and application-level tests. The final generation benchmark uses one shared loop for all test questions.

## 1. Setup and evaluation questions

First, connect to the existing ChromaDB collection and define a small benchmark. **Top-k** means the number of retrieved chunks checked for each question.

In [1]:
import json
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

PROJECT_ROOT = Path.cwd().parent
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
TOP_K = 5
COLLECTION_NAME = "finsolve_documents"

In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = Chroma(
    persist_directory=str(CHROMA_DIR),
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,                                                                                                                                                                                                                                  
)

/home/khushi/Codebasics/Dataset/DS-RPC-01/rag_chatbot_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9485.17it/s]


In [3]:
evaluation_data = [
    {
        "question": "What is FinSolve Technologies' company vision?",
        "expected_sources": ["employee_handbook.md"],
        "type": "fact_lookup",
    },
    {
        "question": "What is the employee referral reward?",
        "expected_sources": ["employee_handbook.md"],
        "type": "fact_lookup",
    },
    {
        "question": "What are the engineering team's responsibilities?",
        "expected_sources": ["engineering_master_doc.md"],
        "type": "fact_lookup",
    },
    {
        "question": "What were the company's marketing results in Q4 2024?",
        "expected_sources": ["market_report_q4_2024.md"],
        "type": "summary",
    },
    {
        "question": "What were the marketing results in Q1 2024?",
        "expected_sources": ["marketing_report_q1_2024.md"],
        "type": "summary",
    },
    {
        "question": "What were the marketing results in Q2 2024?",
        "expected_sources": ["marketing_report_q2_2024.md"],
        "type": "summary",
    },
    {
        "question": "What were the marketing results in Q3 2024?",
        "expected_sources": ["marketing_report_q3_2024.md"],
        "type": "summary",
    },
    {
        "question": "What is included in the financial summary?",
        "expected_sources": ["financial_summary.md"],
        "type": "summary",
    },
    {
        "question": "What are the company's financial results?",
        "expected_sources": ["quarterly_financial_report.md"],
        "type": "summary",
    },
    {
        "question": "What does the employee handbook say about employee referrals?",
        "expected_sources": ["employee_handbook.md"],
        "type": "fact_lookup",
    },

    # HR / structured data
    {
        "question": "How many employees are there?",
        "expected_sources": ["hr_data.csv"],
        "type": "aggregation",
    },
    {
        "question": "What is the average salary?",
        "expected_sources": ["hr_data.csv"],
        "type": "aggregation",
    },

    # Unanswerable
    {
        "question": "What is FinSolve's office location in London?",
        "expected_sources": [],
        "type": "unanswerable",
    },
    {
        "question": "What is FinSolve's CEO's phone number?",
        "expected_sources": [],
        "type": "unanswerable",
    },
]

In [4]:
def evaluate_retrieval(vector_store, evaluation_data, top_k=5):

    results = []

    for item in evaluation_data:

        documents = vector_store.similarity_search(
            item["question"],
            k=top_k,
        )

        retrieved_sources = {
            doc.metadata.get("source")
            for doc in documents
        }

        expected_sources = set(item["expected_sources"])

        if not expected_sources:
            # For unanswerable questions, retrieval itself
            # is not scored as normal source recall.
            hit = None
        else:
            hit = bool(
                expected_sources.intersection(
                    retrieved_sources
                )
            )

        results.append({
            "question": item["question"],
            "type": item["type"],
            "expected_sources": list(expected_sources),
            "retrieved_sources": list(retrieved_sources),
            "retrieval_hit": hit,
        })

    return results

In [5]:
retrieval_results = evaluate_retrieval(
    vector_store,
    evaluation_data,
    TOP_K,
)

In [6]:
for result in retrieval_results:

    if result["retrieval_hit"] is False:

        print("\nQUESTION:")
        print(result["question"])

        print("EXPECTED:")
        print(result["expected_sources"])

        print("RETRIEVED:")
        print(result["retrieved_sources"])

        print("-" * 60)


QUESTION:
What were the marketing results in Q1 2024?
EXPECTED:
['marketing_report_q1_2024.md']
RETRIEVED:
['quarterly_financial_report.md', 'market_report_q4_2024.md']
------------------------------------------------------------


In [7]:
scored_results = [
    r for r in retrieval_results
    if r["retrieval_hit"] is not None
]

hits = sum(
    r["retrieval_hit"]
    for r in scored_results
)

recall_at_5 = hits / len(scored_results)

print(f"Retrieval Recall@5: {recall_at_5:.2%}")

Retrieval Recall@5: 91.67%


## 2. Baseline retrieval: inspect what search returns

This first diagnostic checks a real marketing question and prints the sources found by the initial dense ChromaDB search. Use it to see whether the expected document appears in the top results.

In [8]:
query = "What were the marketing results in Q1 2024?"

print("=" * 70)
print("RETRIEVAL DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# 1. NORMAL RETRIEVAL
# ------------------------------------------------------------

print("\n[1] NORMAL RETRIEVAL")

results = vector_store.similarity_search_with_score(
    query,
    k=10,
)

for i, (doc, score) in enumerate(results, 1):
    print(
        f"{i}. score={score:.4f} | "
        f"department={doc.metadata.get('department')} | "
        f"source={doc.metadata.get('source')}"
    )


# ------------------------------------------------------------
# 2. MARKETING-ONLY RETRIEVAL
# ------------------------------------------------------------

print("\n[2] MARKETING-ONLY RETRIEVAL")

results = vector_store.similarity_search_with_score(
    query,
    k=10,
    filter={"department": "marketing"},
)

for i, (doc, score) in enumerate(results, 1):
    print(
        f"{i}. score={score:.4f} | "
        f"source={doc.metadata.get('source')}"
    )


# ------------------------------------------------------------
# 3. Q1 + MARKETING KEYWORD SEARCH IN STORED DOCUMENTS
# ------------------------------------------------------------

print("\n[3] EXPECTED Q1 MARKETING CHUNKS")

all_data = vector_store.get(
    include=["documents", "metadatas"]
)

found = 0

for document, metadata in zip(
    all_data["documents"],
    all_data["metadatas"],
):

    text = document.lower()

    if (
        metadata.get("department") == "marketing"
        and "q1" in text
        and "2024" in text
    ):
        found += 1

        print(
            f"\n{found}. "
            f"{metadata.get('source')}"
        )

        print(
            f"Section: "
            f"{metadata.get('h1')} | "
            f"{metadata.get('h2')} | "
            f"{metadata.get('h3')}"
        )

        print(document[:500])


# ------------------------------------------------------------
# 4. METADATA CHECK
# ------------------------------------------------------------

print("\n[4] METADATA CHECK")

for metadata in all_data["metadatas"][:5]:
    print(metadata)


# ------------------------------------------------------------
# 5. EMBEDDING RETRIEVAL WITH QUERY VARIATIONS
# ------------------------------------------------------------

print("\n[5] QUERY VARIATION TEST")

queries = [
    "Q1 2024 marketing results",
    "Q1 2024 marketing performance",
    "marketing performance January March 2024",
    "marketing report Q1 2024",
]

for test_query in queries:

    result = vector_store.similarity_search_with_score(
        test_query,
        k=3,
    )

    print(f"\nQuery: {test_query}")

    for i, (doc, score) in enumerate(result, 1):
        print(
            f"{i}. {score:.4f} | "
            f"{doc.metadata.get('source')} | "
            f"{doc.metadata.get('department')}"
        )

RETRIEVAL DIAGNOSTIC

[1] NORMAL RETRIEVAL
1. score=0.2901 | department=finance | source=quarterly_financial_report.md
2. score=0.3099 | department=marketing | source=market_report_q4_2024.md
3. score=0.3619 | department=finance | source=quarterly_financial_report.md
4. score=0.4210 | department=marketing | source=market_report_q4_2024.md
5. score=0.4273 | department=marketing | source=market_report_q4_2024.md
6. score=0.4369 | department=finance | source=quarterly_financial_report.md
7. score=0.4417 | department=marketing | source=marketing_report_q1_2024.md
8. score=0.4436 | department=marketing | source=marketing_report_q2_2024.md
9. score=0.4573 | department=marketing | source=marketing_report_q2_2024.md
10. score=0.4696 | department=marketing | source=market_report_q4_2024.md

[2] MARKETING-ONLY RETRIEVAL
1. score=0.3099 | source=market_report_q4_2024.md
2. score=0.4210 | source=market_report_q4_2024.md
3. score=0.4273 | source=market_report_q4_2024.md
4. score=0.4417 | source=mar

## 3. Improve retrieval with hybrid search

Dense search finds semantically similar text; **BM25** finds lexical/keyword matches. Hybrid retrieval combines both signals so exact terms, names, and phrases are less likely to be missed.

In [9]:
# ============================================================
# HYBRID RETRIEVAL V3
# Dense Chroma + BM25 lexical retrieval
# ============================================================

from collections import defaultdict
from rank_bm25 import BM25Okapi

# ------------------------------------------------------------
# 1. Load all chunks from existing Chroma collection
# ------------------------------------------------------------

data = vector_store.get(include=["documents", "metadatas"])

documents = data["documents"]
metadatas = data["metadatas"]

print(f"Total chunks loaded: {len(documents)}")


# ------------------------------------------------------------
# 2. Build BM25 index
# ------------------------------------------------------------

tokenized_corpus = [
    doc.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created.")


# ------------------------------------------------------------
# 3. Hybrid retrieval
# ------------------------------------------------------------

def hybrid_retrieve(query, k=5, dense_k=10, bm25_k=10):

    # -------------------------
    # Dense retrieval
    # -------------------------
    dense_results = vector_store.similarity_search_with_score(
        query,
        k=dense_k
    )

    # Convert document text -> dense rank
    dense_rank = {}

    for rank, (doc, score) in enumerate(dense_results, start=1):
        dense_rank[doc.page_content] = rank


    # -------------------------
    # BM25 retrieval
    # -------------------------
    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:bm25_k]

    bm25_rank = {
        documents[i]: rank
        for rank, i in enumerate(bm25_indices, start=1)
    }


    # --------------------------------------------------------
    # 4. Reciprocal Rank Fusion (RRF)
    # --------------------------------------------------------

    rrf_scores = defaultdict(float)

    RRF_K = 60

    for doc_text, rank in dense_rank.items():
        rrf_scores[doc_text] += 1 / (RRF_K + rank)

    for doc_text, rank in bm25_rank.items():
        rrf_scores[doc_text] += 1 / (RRF_K + rank)


    # --------------------------------------------------------
    # 5. Sort final hybrid ranking
    # --------------------------------------------------------

    ranked_docs = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for doc_text, score in ranked_docs[:k]:

        idx = documents.index(doc_text)

        results.append({
            "content": doc_text,
            "metadata": metadatas[idx],
            "score": score
        })

    return results


# ============================================================
# 6. Inspect the two previous failure cases
# ============================================================

test_queries = [
    "What is the employee referral reward?",
    "What does the employee handbook say about employee referrals?",
    "What were the marketing results in Q1 2024?"
]

for query in test_queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    results = hybrid_retrieve(query, k=5)

    for i, result in enumerate(results, start=1):

        print(
            f"{i}. score={result['score']:.4f} | "
            f"{result['metadata'].get('source')} | "
            f"{result['metadata'].get('department')}"
        )


# ============================================================
# 7. Evaluate Recall@5
# ============================================================

print("\n" + "=" * 70)
print("V3 HYBRID RETRIEVAL EVALUATION")
print("=" * 70)

correct = 0
total = 0

for item in evaluation_data:

    expected_sources = set(item["expected_sources"])

    # Skip unanswerable questions
    if not expected_sources:
        continue

    total += 1

    results = hybrid_retrieve(
        item["question"],
        k=5
    )

    retrieved_sources = {
        result["metadata"].get("source")
        for result in results
    }

    if expected_sources.intersection(retrieved_sources):
        correct += 1
    else:
        print("\nFAILED:")
        print(item["question"])
        print("Expected:", expected_sources)
        print("Retrieved:", retrieved_sources)

print("\n" + "-" * 70)
print(f"V3 Hybrid Recall@5: {correct / total:.2%}")
print("-" * 70)

Total chunks loaded: 316
BM25 index created.

QUERY: What is the employee referral reward?
1. score=0.0328 | employee_handbook.md | general
2. score=0.0294 | employee_handbook.md | general
3. score=0.0161 | employee_handbook.md | general
4. score=0.0161 | employee_handbook.md | general
5. score=0.0159 | marketing_report_q1_2024.md | marketing

QUERY: What does the employee handbook say about employee referrals?
1. score=0.0328 | employee_handbook.md | general
2. score=0.0323 | employee_handbook.md | general
3. score=0.0317 | employee_handbook.md | general
4. score=0.0303 | employee_handbook.md | general
5. score=0.0299 | employee_handbook.md | general

QUERY: What were the marketing results in Q1 2024?
1. score=0.0315 | market_report_q4_2024.md | marketing
2. score=0.0308 | market_report_q4_2024.md | marketing
3. score=0.0303 | marketing_report_q1_2024.md | marketing
4. score=0.0297 | market_report_q4_2024.md | marketing
5. score=0.0164 | quarterly_financial_report.md | finance

V3 HYB

In [10]:
# ============================================================
# V3 RETRIEVAL QUALITY DIAGNOSTIC
# ============================================================

print("=" * 80)
print("V3 RETRIEVAL QUALITY DIAGNOSTIC")
print("=" * 80)

answerable_questions = [
    item for item in evaluation_data
    if item["expected_sources"]
]

for idx, item in enumerate(answerable_questions, start=1):

    query = item["question"]
    expected_sources = set(item["expected_sources"])

    results = hybrid_retrieve(query, k=5)

    retrieved_sources = [
        r["metadata"].get("source")
        for r in results
    ]

    expected_found = bool(
        expected_sources.intersection(retrieved_sources)
    )

    print("\n" + "=" * 80)
    print(f"[{idx}/{len(answerable_questions)}]")
    print("QUESTION:", query)
    print("EXPECTED SOURCE:", expected_sources)
    print("EXPECTED SOURCE FOUND:", expected_found)

    print("\nTOP 5 RESULTS:")

    for rank, result in enumerate(results, start=1):

        source = result["metadata"].get("source")
        score = result["score"]

        content = result["content"].replace("\n", " ")

        # Show only beginning of chunk
        preview = content[:350]

        marker = " <-- EXPECTED" if source in expected_sources else ""

        print(
            f"\n{rank}. score={score:.4f} | "
            f"{source}{marker}"
        )

        print("   ", preview)

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

V3 RETRIEVAL QUALITY DIAGNOSTIC

[1/12]
QUESTION: What is FinSolve Technologies' company vision?
EXPECTED SOURCE: {'employee_handbook.md'}
EXPECTED SOURCE FOUND: True

TOP 5 RESULTS:

1. score=0.0325 | employee_handbook.md <-- EXPECTED
    ## Welcome & Introduction   ### Company Vision and Mission At FinSolve Technologies, our vision is to empower businesses and individuals through innovative technology solutions. Our mission is to deliver high-quality, sustainable products and services that create value for our stakeholders.

2. score=0.0320 | engineering_master_doc.md
    # FinSolve Technologies Engineering Document   ## 1. Introduction   ### 1.1 Company Overview FinSolve Technologies is a leading FinTech company headquartered in Bangalore, India, with operations across North America, Europe, and Asia-Pacific. Founded in 2018, FinSolve provides innovative financial solutions, including digital banking, payment proce

3. score=0.0320 | employee_handbook.md <-- EXPECTED
    ### Company

## 4. Structured HR data checks

HR questions such as employee counts and average salary need calculations over the CSV, not only semantic search over document chunks. These cells inspect the dataset shape and test basic aggregates without printing individual employee records.

### 4.1 Understand the HR data route

In [11]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
HR_FILE = PROJECT_ROOT / "data" / "hr" / "hr_data.csv"

hr_df = pd.read_csv(HR_FILE)

# Privacy-safe inspection: print dimensions and column names only
# Do NOT print individual rows or personal employee records
print(f"Dataset shape: {hr_df.shape} (rows, columns)")
print("Columns:", hr_df.columns.tolist())
print("Privacy check: Employee-level PII suppressed.")

Dataset shape: (100, 15) (rows, columns)
Columns: ['employee_id', 'full_name', 'role', 'department', 'email', 'location', 'date_of_birth', 'date_of_joining', 'manager_id', 'salary', 'leave_balance', 'leaves_taken', 'attendance_pct', 'performance_rating', 'last_review_date']
Privacy check: Employee-level PII suppressed.


In [12]:
# Employee count
employee_count = hr_df["employee_id"].nunique()

# Average salary
average_salary = hr_df["salary"].mean()

print("Employee count:", employee_count)
print("Average salary:", round(average_salary, 2))

Employee count: 100
Average salary: 1187565.31


## 5. Retrieval and context-strategy evaluation

Compare retrieval/context strategies using the metrics implemented below. **Recall@k** checks whether expected sources appear among the first *k* results; precision and MRR provide additional views of result quality. Reranking and source-diversity strategies are experiments, not assumed improvements—the reported metrics should guide the choice.

In [13]:
# ============================================================
# RAG AUGMENTATION — MULTI-STRATEGY EVALUATION
# ============================================================
# Strategies:
# A. Baseline Hybrid Retrieval
# B. Deduplication + Source Diversity
# C. LLM Reranking + Safe Fallback
#
# Evaluates:
# Recall, Precision, MRR, duplicates, context size,
# reranking failures, recall loss, precision improvement
# ============================================================

import sys
from pathlib import Path
import json
import re
import pandas as pd

# ============================================================
# 1. PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = Path.cwd()

APP_DIR = PROJECT_ROOT / "app"

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))


# ============================================================
# 2. IMPORTS
# ============================================================

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

from retriever import HybridRetriever

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from config import (
    CHROMA_DIR,
    COLLECTION_NAME,
    EMBEDDING_MODEL,
    GROQ_MODEL,
)


# ============================================================
# 3. ENV + LLM
# ============================================================

load_dotenv(PROJECT_ROOT / ".env")

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_tokens=1200,
)


# ============================================================
# 4. QUERY UNDERSTANDING
# ============================================================

router_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are the query-understanding layer of a company RAG chatbot.

Understand the user's meaning rather than exact keywords.

Return ONLY valid JSON.

The JSON must have this structure:

{{
  "intent": "...",
  "search_query": "..."
}}

Allowed intents:

conversation
company_information
employee_count
salary_information
hr_information
finance_information
marketing_information
engineering_information
general_company_information
unknown

Examples:

"hi"
=> conversation

"how are you?"
=> conversation

"who are you?"
=> conversation

"what does the company do?"
=> company_information

"how much money does the company make?"
=> finance_information

"how many people work here?"
=> employee_count

"what do employees usually earn?"
=> salary_information

"tell me about marketing"
=> marketing_information

"what does engineering handle?"
=> engineering_information

For company questions, rewrite the question into
a clear retrieval query.

Do not answer the question.
Only return JSON.
"""
        ),
        (
            "human",
            "{question}"
        ),
    ]
)


def understand_query(question):

    messages = router_prompt.format_messages(
        question=question
    )

    response = llm.invoke(messages)

    text = response.content.strip()

    text = re.sub(
        r"```json|```",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    try:
        return json.loads(text)

    except Exception:

        return {
            "intent": "unknown",
            "search_query": question,
        }


# ============================================================
# 5. VECTOR STORE + HYBRID RETRIEVER
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
)

retriever = HybridRetriever(vector_store)


# ============================================================
# 6. EVALUATION DATASET
# ============================================================

EVALUATION_SET = [

    {
        "question": "What does the company do?",
        "expected": ["employee_handbook.md"],
        "answerable": True,
    },

    {
        "question": "What is the company's vision?",
        "expected": ["employee_handbook.md"],
        "answerable": True,
    },

    {
        "question": "How does the employee referral program work?",
        "expected": ["employee_handbook.md"],
        "answerable": True,
    },

    {
        "question": "What does the engineering team handle?",
        "expected": ["engineering_master_doc.md"],
        "answerable": True,
    },

    {
        "question": "What is the software development lifecycle?",
        "expected": ["engineering_master_doc.md"],
        "answerable": True,
    },

    {
        "question": "What were the company's financial results?",
        "expected": ["quarterly_financial_report.md"],
        "answerable": True,
    },

    {
        "question": "What is the company's revenue?",
        "expected": [
            "quarterly_financial_report.md",
            "financial_summary.md",
        ],
        "answerable": True,
    },

    {
        "question": "Tell me about Q1 marketing performance.",
        "expected": ["marketing_report_q1_2024.md"],
        "answerable": True,
    },

    {
        "question": "Tell me about Q2 marketing performance.",
        "expected": ["marketing_report_q2_2024.md"],
        "answerable": True,
    },

    {
        "question": "Tell me about Q3 marketing performance.",
        "expected": ["marketing_report_q3_2024.md"],
        "answerable": True,
    },

    {
        "question": "Tell me about Q4 marketing performance.",
        "expected": ["market_report_q4_2024.md"],
        "answerable": True,
    },

    {
        "question": "What does the marketing department do?",
        "expected": [
            "marketing_report_2024.md",
            "market_report_q4_2024.md",
            "marketing_report_q1_2024.md",
            "marketing_report_q2_2024.md",
            "marketing_report_q3_2024.md",
        ],
        "answerable": True,
    },

    {
        "question": "How many employees work in the company?",
        "expected": ["hr_data.csv"],
        "answerable": True,
    },

    {
        "question": "What is the average employee salary?",
        "expected": ["hr_data.csv"],
        "answerable": True,
    },

    {
        "question": "What is the CEO's phone number?",
        "expected": [],
        "answerable": False,
    },

    {
        "question": "What is the company's stock price today?",
        "expected": [],
        "answerable": False,
    },

    {
        "question": "What is the weather today?",
        "expected": [],
        "answerable": False,
    },

    {
        "question": "Hi",
        "expected": [],
        "answerable": False,
    },

    {
        "question": "How are you?",
        "expected": [],
        "answerable": False,
    },
]


# ============================================================
# 7. METRICS
# ============================================================

def source(result):
    return result["metadata"].get(
        "source",
        "unknown"
    )


def unique_sources(results):
    return list(
        dict.fromkeys(
            source(result)
            for result in results
        )
    )


def recall_at_k(results, expected):

    if not expected:
        return None

    retrieved = set(
        source(result)
        for result in results
    )

    expected = set(expected)

    return len(
        retrieved & expected
    ) / len(expected)


def precision_at_k(results, expected):

    if not results:
        return 0

    expected = set(expected)

    relevant = sum(
        source(result) in expected
        for result in results
    )

    return relevant / len(results)


def mrr(results, expected):

    expected = set(expected)

    for rank, result in enumerate(
        results,
        start=1
    ):

        if source(result) in expected:
            return 1 / rank

    return 0


def duplicate_rate(results):

    if not results:
        return 0

    documents = [
        result["document"]
        for result in results
    ]

    return (
        len(documents)
        - len(set(documents))
    ) / len(documents)


def context_chars(results):

    return sum(
        len(result["document"])
        for result in results
    )


# ============================================================
# 8. STRATEGY B
# DEDUPLICATION + SOURCE DIVERSITY
# ============================================================

def strategy_b(results, k=5):

    unique = []
    seen = set()

    for result in results:

        document = result["document"]

        if document not in seen:

            unique.append(result)
            seen.add(document)

    selected = []
    used_sources = set()

    # First pass: source diversity
    for result in unique:

        src = source(result)

        if src not in used_sources:

            selected.append(result)
            used_sources.add(src)

        if len(selected) >= k:
            break

    # Second pass: fill remaining positions
    for result in unique:

        if len(selected) >= k:
            break

        if result not in selected:
            selected.append(result)

    return selected[:k]


# ============================================================
# 9. STRATEGY C — LLM RERANKER
# ============================================================
# IMPORTANT:
# No structured output / tool calling here.
# We use normal text JSON to avoid the previous Groq
# "rerankResult" tool validation problem.
# ============================================================

rerank_prompt = """
You are a RAG reranking system.

Evaluate how relevant each candidate chunk is
to the user's question.

Return ONLY a JSON array.

Example:

[
  {{"index": 1, "score": 10}},
  {{"index": 2, "score": 3}},
  {{"index": 3, "score": 0}}
]

Scoring:

10 = directly answers the question
8-9 = highly relevant
6-7 = useful supporting information
4-5 = weakly related
1-3 = mostly irrelevant
0 = irrelevant

Judge the actual content.

Do not rank a chunk highly only because
its filename contains a matching keyword.

Question:
{question}

Candidates:
{candidates}
"""


def parse_json_array(text):

    text = text.strip()

    text = re.sub(
        r"```json|```",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    match = re.search(
        r"\[[\s\S]*\]",
        text
    )

    if not match:
        return None

    try:

        data = json.loads(
            match.group(0)
        )

        if isinstance(data, list):
            return data

    except Exception:

        return None

    return None


def strategy_c(
    question,
    results,
    k=5
):

    # Deduplicate before LLM reranking
    unique = []
    seen = set()

    for result in results:

        document = result["document"]

        if document not in seen:

            unique.append(result)
            seen.add(document)

    candidates = []

    for i, result in enumerate(
        unique,
        start=1
    ):

        candidates.append(
            f"""
CANDIDATE {i}

Source: {source(result)}

Department: {
    result["metadata"].get(
        "department",
        "unknown"
    )
}

Content:
{result["document"]}
"""
        )

    candidates_text = "\n".join(
        candidates
    )

    prompt = rerank_prompt.format(
        question=question,
        candidates=candidates_text,
    )

    try:

        response = llm.invoke(prompt)

        rankings = parse_json_array(
            response.content
        )

        if not rankings:

            return strategy_b(
                results,
                k
            ), "fallback"

        scored = []

        for item in rankings:

            try:

                index = int(
                    item["index"]
                )

                score = float(
                    item["score"]
                )

            except Exception:

                continue

            if 1 <= index <= len(unique):

                scored.append(
                    (
                        score,
                        unique[index - 1]
                    )
                )

        if not scored:

            return strategy_b(
                results,
                k
            ), "fallback"

        scored.sort(
            key=lambda x: x[0],
            reverse=True
        )

        reranked = [
            item[1]
            for item in scored
        ]

        return reranked[:k], "llm"

    except Exception as e:

        return (
            strategy_b(results, k),
            f"fallback: {str(e)[:120]}"
        )


# ============================================================
# 10. RUN EVALUATION
# ============================================================

records = []
details = []

print("=" * 100)
print("RUNNING 3 AUGMENTATION STRATEGIES")
print("=" * 100)


for item in EVALUATION_SET:

    question = item["question"]
    expected = item["expected"]
    answerable = item["answerable"]


    # --------------------------------------------------------
    # QUERY UNDERSTANDING
    # --------------------------------------------------------

    try:

        parsed = understand_query(
            question
        )

    except Exception as e:

        records.append(
            {
                "question": question,
                "strategy": "ALL",
                "status": "ROUTER_ERROR",
                "error": str(e),
            }
        )

        continue


    intent = parsed.get(
        "intent",
        "unknown"
    )

    search_query = parsed.get(
        "search_query",
        question
    )


    # --------------------------------------------------------
    # CONVERSATION
    # --------------------------------------------------------

    if intent == "conversation":
        continue


    # --------------------------------------------------------
    # RETRIEVAL
    # --------------------------------------------------------

    try:

        retrieved = retriever.retrieve(
            search_query
        )

    except Exception as e:

        records.append(
            {
                "question": question,
                "strategy": "ALL",
                "status": "RETRIEVAL_ERROR",
                "error": str(e),
            }
        )

        continue


    # --------------------------------------------------------
    # THREE STRATEGIES
    # --------------------------------------------------------

    baseline = retrieved[:5]

    smart = strategy_b(
        retrieved,
        k=5
    )

    reranked, rerank_status = strategy_c(
        question,
        retrieved,
        k=5
    )


    strategies = {
        "A_BASELINE": baseline,
        "B_DEDUP_DIVERSITY": smart,
        "C_LLM_RERANK": reranked,
    }


    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    for name, selected in strategies.items():

        records.append(
            {
                "question": question,
                "intent": intent,
                "search_query": search_query,
                "strategy": name,

                "recall": (
                    recall_at_k(
                        selected,
                        expected
                    )
                    if answerable
                    else None
                ),

                "precision": (
                    precision_at_k(
                        selected,
                        expected
                    )
                    if answerable
                    else None
                ),

                "mrr": (
                    mrr(
                        selected,
                        expected
                    )
                    if answerable
                    else None
                ),

                "duplicates": duplicate_rate(
                    selected
                ),

                "context_chars": context_chars(
                    selected
                ),

                "sources": unique_sources(
                    selected
                ),

                "rerank_status": (
                    rerank_status
                    if name == "C_LLM_RERANK"
                    else ""
                ),
            }
        )


    details.append(
        {
            "question": question,
            "expected": expected,
            "baseline": baseline,
            "smart": smart,
            "reranked": reranked,
            "rerank_status": rerank_status,
        }
    )


# ============================================================
# 11. QUESTION LEVEL RESULTS
# ============================================================

results_df = pd.DataFrame(records)

print("\n")
print("=" * 100)
print("QUESTION-LEVEL RESULTS")
print("=" * 100)

display(
    results_df[
        [
            "question",
            "strategy",
            "recall",
            "precision",
            "mrr",
            "duplicates",
            "context_chars",
            "rerank_status",
        ]
    ]
)


# ============================================================
# 12. STRATEGY SUMMARY
# ============================================================

answerable_results = results_df[
    results_df["recall"].notna()
]

summary = (
    answerable_results
    .groupby("strategy")
    .agg(
        Recall=("recall", "mean"),
        Precision=("precision", "mean"),
        MRR=("mrr", "mean"),
        Duplicate_Rate=("duplicates", "mean"),
        Context_Chars=("context_chars", "mean"),
    )
    .reset_index()
)


print("\n")
print("=" * 100)
print("STRATEGY COMPARISON")
print("=" * 100)

display(summary)


# ============================================================
# 13. STRATEGY SCORE
# ============================================================

ranking = summary.copy()

max_context = ranking["Context_Chars"].max()

if max_context == 0:
    context_efficiency = 1
else:
    context_efficiency = (
        1 -
        ranking["Context_Chars"]
        / max_context
    )

ranking["Overall_Score"] = (
    ranking["Recall"] * 0.40
    + ranking["Precision"] * 0.30
    + ranking["MRR"] * 0.20
    + (1 - ranking["Duplicate_Rate"]) * 0.05
    + context_efficiency * 0.05
)

ranking = ranking.sort_values(
    "Overall_Score",
    ascending=False
)


print("\n")
print("=" * 100)
print("OVERALL STRATEGY RANKING")
print("=" * 100)

display(ranking)


# ============================================================
# 14. BEFORE vs AFTER
# ============================================================

baseline_row = summary[
    summary["strategy"] == "A_BASELINE"
].iloc[0]

print("\n")
print("=" * 100)
print("IMPROVEMENT OVER BASELINE")
print("=" * 100)

for _, row in summary.iterrows():

    print(
        f"""
{row['strategy']}

Recall Change:
    {row['Recall'] - baseline_row['Recall']:+.1%}

Precision Change:
    {row['Precision'] - baseline_row['Precision']:+.1%}

MRR Change:
    {row['MRR'] - baseline_row['MRR']:+.3f}

Duplicate Change:
    {row['Duplicate_Rate'] - baseline_row['Duplicate_Rate']:+.1%}

Context Size Change:
    {row['Context_Chars'] - baseline_row['Context_Chars']:+.0f} chars
"""
    )


# ============================================================
# 15. RECALL SAFETY CHECK
# ============================================================

print("=" * 100)
print("RECALL SAFETY CHECK")
print("=" * 100)

baseline_recall = baseline_row["Recall"]

for _, row in summary.iterrows():

    difference = (
        row["Recall"]
        - baseline_recall
    )

    if difference < 0:

        print(
            f"❌ {row['strategy']}: "
            f"Recall decreased by "
            f"{abs(difference):.1%}"
        )

    else:

        print(
            f"✅ {row['strategy']}: "
            f"Recall preserved/improved "
            f"({difference:+.1%})"
        )


# ============================================================
# 16. IMPORTANT QUESTION ANALYSIS
# ============================================================

important_questions = [
    "What does the marketing department do?",
    "How many employees work in the company?",
    "What is the company's revenue?",
    "What is the software development lifecycle?",
]


print("\n")
print("=" * 100)
print("IMPORTANT QUESTION ANALYSIS")
print("=" * 100)


for detail in details:

    question = detail["question"]

    if question not in important_questions:
        continue

    print("\n")
    print("-" * 100)

    print("QUESTION:", question)
    print("EXPECTED:", detail["expected"])

    for label, key in [
        ("A_BASELINE", "baseline"),
        ("B_DEDUP_DIVERSITY", "smart"),
        ("C_LLM_RERANK", "reranked"),
    ]:

        selected = detail[key]

        print(f"\n{label}")

        print(
            "Sources:",
            unique_sources(selected)
        )

        print(
            "Recall:",
            recall_at_k(
                selected,
                detail["expected"]
            )
        )

        print(
            "Precision:",
            precision_at_k(
                selected,
                detail["expected"]
            )
        )

        print(
            "MRR:",
            mrr(
                selected,
                detail["expected"]
            )
        )

        print(
            "Context:",
            context_chars(selected),
            "chars"
        )


# ============================================================
# 17. QUESTIONS WHERE STRATEGY LOST RECALL
# ============================================================

print("\n")
print("=" * 100)
print("RECALL LOSS / PROBLEM QUESTIONS")
print("=" * 100)

problem_found = False

for detail in details:

    expected = detail["expected"]

    if not expected:
        continue

    baseline_recall_value = recall_at_k(
        detail["baseline"],
        expected
    )

    for label, key in [
        ("B_DEDUP_DIVERSITY", "smart"),
        ("C_LLM_RERANK", "reranked"),
    ]:

        current_recall = recall_at_k(
            detail[key],
            expected
        )

        if (
            current_recall
            < baseline_recall_value
        ):

            problem_found = True

            print(
                f"""
Question:
{detail['question']}

Strategy:
{label}

Baseline Recall:
{baseline_recall_value:.1%}

Current Recall:
{current_recall:.1%}

Expected:
{expected}

Baseline Sources:
{unique_sources(detail['baseline'])}

Current Sources:
{unique_sources(detail[key])}
"""
            )


if not problem_found:

    print("No strategy lost recall compared with baseline.")


# ============================================================
# 18. RERANKER FALLBACK STATUS
# ============================================================

print("\n")
print("=" * 100)
print("LLM RERANKER STATUS")
print("=" * 100)

fallback_count = sum(
    1
    for detail in details
    if str(
        detail["rerank_status"]
    ).startswith("fallback")
)

llm_success_count = sum(
    1
    for detail in details
    if detail["rerank_status"] == "llm"
)

print(
    "LLM reranker successful:",
    llm_success_count
)

print(
    "Fallback used:",
    fallback_count
)


# ============================================================
# 19. FINAL DECISION
# ============================================================

best = ranking.iloc[0]

best_strategy = best["strategy"]
best_recall = best["Recall"]
best_precision = best["Precision"]

baseline_precision = baseline_row["Precision"]


print("\n")
print("=" * 100)
print("FINAL AUGMENTATION DECISION")
print("=" * 100)


if (
    best_recall >= baseline_recall
    and
    best_precision > baseline_precision
):

    print(
        f"""
✅ AUGMENTATION READY

Best Strategy:
{best_strategy}

Recall:
{best_recall:.1%}

Precision:
{best_precision:.1%}

Baseline Recall:
{baseline_recall:.1%}

Baseline Precision:
{baseline_precision:.1%}

The augmentation stage improved context quality
without losing retrieval recall.

NEXT:
Move to GENERATION + GENERATION EVALUATION.
"""
    )

elif best_recall >= baseline_recall:

    print(
        f"""
⚠️ AUGMENTATION ACCEPTABLE

Best Strategy:
{best_strategy}

Recall:
{best_recall:.1%}

Precision:
{best_precision:.1%}

Retrieval evidence is preserved.

Review the question-level results once,
then move to generation.
"""
    )

else:

    print(
        f"""
❌ AUGMENTATION NOT READY

Best Strategy:
{best_strategy}

Recall:
{best_recall:.1%}

Baseline Recall:
{baseline_recall:.1%}

An augmentation strategy is losing evidence.
Do not move to generation yet.
"""
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9429.98it/s]


RUNNING 3 AUGMENTATION STRATEGIES


QUESTION-LEVEL RESULTS


,question,strategy,recall,precision,mrr,duplicates,context_chars,rerank_status
0,What does the company do?,A_BASELINE,1.0,0.8,1.0,0.0,1100,
1,What does the company do?,B_DEDUP_DIVERSITY,1.0,0.8,1.0,0.0,1100,
2,What does the company do?,C_LLM_RERANK,1.0,0.8,0.5,0.0,1100,llm
3,What is the company's vision?,A_BASELINE,1.0,0.8,1.0,0.0,1038,
4,What is the company's vision?,B_DEDUP_DIVERSITY,1.0,0.8,1.0,0.0,1038,
5,What is the company's vision?,C_LLM_RERANK,1.0,0.8,1.0,0.0,1038,llm
6,How does the employee referral program work?,A_BASELINE,1.0,0.8,1.0,0.0,1135,
7,How does the employee referral program work?,B_DEDUP_DIVERSITY,1.0,0.8,1.0,0.0,1135,
8,How does the employee referral program work?,C_LLM_RERANK,1.0,0.8,1.0,0.0,1135,llm
9,What does the engineering team handle?,A_BASELINE,1.0,1.0,1.0,0.0,2077,




STRATEGY COMPARISON


,strategy,Recall,Precision,MRR,Duplicate_Rate,Context_Chars
0,A_BASELINE,0.942857,0.642857,0.928571,0.0,2407.071429
1,B_DEDUP_DIVERSITY,0.942857,0.642857,0.928571,0.0,2407.071429
2,C_LLM_RERANK,0.942857,0.642857,0.964286,0.0,2407.071429




OVERALL STRATEGY RANKING


,strategy,Recall,Precision,MRR,Duplicate_Rate,Context_Chars,Overall_Score
2,C_LLM_RERANK,0.942857,0.642857,0.964286,0.0,2407.071429,0.812857
0,A_BASELINE,0.942857,0.642857,0.928571,0.0,2407.071429,0.805714
1,B_DEDUP_DIVERSITY,0.942857,0.642857,0.928571,0.0,2407.071429,0.805714




IMPROVEMENT OVER BASELINE

A_BASELINE

Recall Change:
    +0.0%

Precision Change:
    +0.0%

MRR Change:
    +0.000

Duplicate Change:
    +0.0%

Context Size Change:
    +0 chars


B_DEDUP_DIVERSITY

Recall Change:
    +0.0%

Precision Change:
    +0.0%

MRR Change:
    +0.000

Duplicate Change:
    +0.0%

Context Size Change:
    +0 chars


C_LLM_RERANK

Recall Change:
    +0.0%

Precision Change:
    +0.0%

MRR Change:
    +0.036

Duplicate Change:
    +0.0%

Context Size Change:
    +0 chars

RECALL SAFETY CHECK
✅ A_BASELINE: Recall preserved/improved (+0.0%)
✅ B_DEDUP_DIVERSITY: Recall preserved/improved (+0.0%)
✅ C_LLM_RERANK: Recall preserved/improved (+0.0%)


IMPORTANT QUESTION ANALYSIS


----------------------------------------------------------------------------------------------------
QUESTION: What is the software development lifecycle?
EXPECTED: ['engineering_master_doc.md']

A_BASELINE
Sources: ['engineering_master_doc.md', 'employee_handbook.md']
Recall: 1.0
Precisio

## 6. Generation evaluation

One evaluation loop runs the same pipeline for all 14 benchmark questions: hybrid retrieval → context building → LLM answer → LLM judge → metrics. The questions are test data, not separate production logic. No question-specific diagnostic cells are needed here.

In [14]:
# ============================================================
# CHAPTER 4 — GENERATION + GENERATION EVALUATION
# FIXED FOR YOUR HybridRetriever(vector_store) IMPLEMENTATION
# ============================================================

import sys
import json
import re
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. PROJECT SETUP
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

APP_DIR = PROJECT_ROOT / "app"

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

from config import (
    CHROMA_DIR,
    COLLECTION_NAME,
    EMBEDDING_MODEL,
    GROQ_MODEL,
    TEMPERATURE,
    MAX_TOKENS,
)

from retriever import HybridRetriever
from generator import create_llm

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# ------------------------------------------------------------
# 2. INITIALIZE EMBEDDINGS + CHROMA + RETRIEVER + LLM
# ------------------------------------------------------------

print("=" * 100)
print("GENERATION EVALUATION")
print("=" * 100)

print("\nLoading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

print("Loading Chroma vector store...")

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

print("Initializing hybrid retriever...")

retriever = HybridRetriever(
    vector_store=vector_store
)

print("Initializing Groq LLM...")

llm = create_llm()

print("\nInitialization complete.")
print(f"LLM       : {GROQ_MODEL}")
print(f"Embeddings: {EMBEDDING_MODEL}")
print("Retriever : Dense + BM25 + RRF")


# ------------------------------------------------------------
# 3. EVALUATION DATASET
# ------------------------------------------------------------

EVALUATION_SET = [

    {
        "question": "What is the software development lifecycle?",
        "answerable": True,
        "expected_sources": ["engineering_master_doc.md"],
    },

    {
        "question": "What is the company's revenue?",
        "answerable": True,
        "expected_sources": [
            "quarterly_financial_report.md",
            "financial_summary.md",
        ],
    },

    {
        "question": "What does the marketing department do?",
        "answerable": True,
        "expected_sources": [
            "market_report_q4_2024.md",
            "marketing_report_2024.md",
        ],
    },

    {
        "question": "How many employees work in the company?",
        "answerable": True,
        "expected_sources": ["hr_data.csv"],
    },

    {
        "question": "What is the average employee salary?",
        "answerable": True,
        "expected_sources": ["hr_data.csv"],
    },

    {
        "question": "What does the engineering team handle?",
        "answerable": True,
        "expected_sources": ["engineering_master_doc.md"],
    },

    {
        "question": "What is the employee referral reward?",
        "answerable": True,
        "expected_sources": ["employee_handbook.md"],
    },

    {
        "question": "What were the company's Q1 marketing results?",
        "answerable": True,
        "expected_sources": ["marketing_report_q1_2024.md"],
    },

    {
        "question": "What were the company's Q2 marketing results?",
        "answerable": True,
        "expected_sources": ["marketing_report_q2_2024.md"],
    },

    {
        "question": "What were the company's Q3 marketing results?",
        "answerable": True,
        "expected_sources": ["marketing_report_q3_2024.md"],
    },

    {
        "question": "What were the Q4 marketing results?",
        "answerable": True,
        "expected_sources": ["market_report_q4_2024.md"],
    },

    {
        "question": "What is the CEO's phone number?",
        "answerable": False,
        "expected_sources": [],
    },

    {
        "question": "What is the company's stock price today?",
        "answerable": False,
        "expected_sources": [],
    },

    {
        "question": "What is the weather in Mumbai today?",
        "answerable": False,
        "expected_sources": [],
    },
]


# ------------------------------------------------------------
# 4. GENERATION PROMPT
# ------------------------------------------------------------

GENERATION_PROMPT = """
You are an internal assistant for FinSolve Technologies.

Answer the user's question ONLY using the supplied company context.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent numbers, names, dates, or facts.
3. If the context does not contain enough evidence, say:
"The available company documents do not contain enough information to answer this."
4. Answer the question directly.
5. When useful, mention the source file.
6. Combine relevant information from multiple sources when necessary.
7. Ignore irrelevant retrieved documents.
8. Never assume information that is not explicitly supported by the context.

COMPANY CONTEXT:

{context}

USER QUESTION:

{question}

ANSWER:
"""


# ------------------------------------------------------------
# 5. CONTEXT BUILDER
# ------------------------------------------------------------

def build_context(results):

    parts = []

    for i, result in enumerate(results, 1):

        metadata = result.get("metadata", {})

        source = metadata.get(
            "source",
            "unknown"
        )

        department = metadata.get(
            "department",
            "unknown"
        )

        sections = []

        for key in ["h1", "h2", "h3", "h4"]:

            value = metadata.get(key)

            if value:
                sections.append(str(value))

        section = " | ".join(sections)

        parts.append(
            f"""
SOURCE {i}

File: {source}
Department: {department}
Section: {section}

CONTENT:
{result["document"]}
"""
        )

    return "\n".join(parts)


# ------------------------------------------------------------
# 6. GENERATE ANSWER
# ------------------------------------------------------------

def generate_answer(question, results):

    context = build_context(results)

    prompt = GENERATION_PROMPT.format(
        context=context,
        question=question,
    )

    response = llm.invoke(prompt)

    return response.content.strip()


# ------------------------------------------------------------
# 7. LLM GENERATION JUDGE
# ------------------------------------------------------------

JUDGE_PROMPT = """
You are evaluating a RAG system answer.

QUESTION:
{question}

AVAILABLE CONTEXT:
{context}

GENERATED ANSWER:
{answer}

EXPECTED ANSWERABILITY:
{answerable}

Evaluate the answer.

1. relevance:
Does the answer directly address the question?
0-10.

2. faithfulness:
Are the claims supported by the supplied context?
0-10.

3. correctness:
Is the answer factually correct according to the context?
0-10.

4. hallucination:
10 = no hallucination.
0 = severe unsupported hallucination.

5. refusal_correct:
If the question is unanswerable, did the assistant correctly refuse?
If the question is answerable, did it avoid incorrectly refusing?
0-10.

Return ONLY valid JSON.

{{
    "relevance": 0,
    "faithfulness": 0,
    "correctness": 0,
    "hallucination": 0,
    "refusal_correct": 0,
    "reason": "short explanation"
}}
"""


def evaluate_answer(
    question,
    context,
    answer,
    answerable,
):

    prompt = JUDGE_PROMPT.format(
        question=question,
        context=context,
        answer=answer,
        answerable=answerable,
    )

    try:

        response = llm.invoke(prompt)

        raw = response.content.strip()

        raw = re.sub(
            r"```json\s*|\s*```",
            "",
            raw,
            flags=re.IGNORECASE,
        ).strip()

        match = re.search(
            r"\{.*\}",
            raw,
            re.DOTALL,
        )

        if not match:
            raise ValueError(
                "LLM did not return JSON"
            )

        data = json.loads(
            match.group()
        )

        return {
            "relevance": float(
                data.get("relevance", 0)
            ),
            "faithfulness": float(
                data.get("faithfulness", 0)
            ),
            "correctness": float(
                data.get("correctness", 0)
            ),
            "hallucination": float(
                data.get("hallucination", 0)
            ),
            "refusal_correct": float(
                data.get("refusal_correct", 0)
            ),
            "reason": data.get(
                "reason",
                ""
            ),
        }

    except Exception as e:

        return {
            "relevance": 0,
            "faithfulness": 0,
            "correctness": 0,
            "hallucination": 0,
            "refusal_correct": 0,
            "reason": (
                f"Evaluation error: {e}"
            ),
        }


# ------------------------------------------------------------
# 8. SOURCE COVERAGE
# ------------------------------------------------------------

def source_coverage(
    results,
    expected_sources,
):

    if not expected_sources:
        return 1.0

    retrieved_sources = {
        result.get(
            "metadata",
            {}
        ).get(
            "source",
            ""
        )
        for result in results
    }

    found = sum(
        1
        for source in expected_sources
        if source in retrieved_sources
    )

    return found / len(
        expected_sources
    )


# ------------------------------------------------------------
# 9. RUN EVALUATION
# ------------------------------------------------------------

results_all = []

for idx, item in enumerate(
    EVALUATION_SET,
    1
):

    question = item["question"]

    print("\n")
    print("=" * 100)
    print(
        f"[{idx}/{len(EVALUATION_SET)}] "
        f"{question}"
    )
    print("=" * 100)

    try:

        # ----------------------------------------------------
        # RETRIEVAL
        # ----------------------------------------------------

        retrieved = retriever.retrieve(
            question,
            k=5,
        )

        # ----------------------------------------------------
        # CONTEXT
        # ----------------------------------------------------

        context = build_context(
            retrieved
        )

        # ----------------------------------------------------
        # GENERATION
        # ----------------------------------------------------

        answer = generate_answer(
            question,
            retrieved,
        )

        # ----------------------------------------------------
        # SOURCE COVERAGE
        # ----------------------------------------------------

        coverage = source_coverage(
            retrieved,
            item["expected_sources"],
        )

        # ----------------------------------------------------
        # GENERATION EVALUATION
        # ----------------------------------------------------

        evaluation = evaluate_answer(
            question=question,
            context=context,
            answer=answer,
            answerable=item["answerable"],
        )

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        print("\nANSWER:")
        print(answer)

        print("\nRETRIEVED SOURCES:")

        for rank, result in enumerate(
            retrieved,
            1
        ):

            source = result.get(
                "metadata",
                {}
            ).get(
                "source",
                "unknown"
            )

            print(
                f"  {rank}. {source}"
            )

        print("\nGENERATION METRICS:")

        print(
            f"  Relevance       : "
            f"{evaluation['relevance']:.1f}/10"
        )

        print(
            f"  Faithfulness    : "
            f"{evaluation['faithfulness']:.1f}/10"
        )

        print(
            f"  Correctness     : "
            f"{evaluation['correctness']:.1f}/10"
        )

        print(
            f"  Hallucination   : "
            f"{evaluation['hallucination']:.1f}/10"
        )

        print(
            f"  Refusal Correct : "
            f"{evaluation['refusal_correct']:.1f}/10"
        )

        print(
            f"  Source Coverage : "
            f"{coverage * 100:.1f}%"
        )

        print(
            f"\nJudge Reason: "
            f"{evaluation['reason']}"
        )

        results_all.append({
            "question": question,
            "answer": answer,
            "answerable": item["answerable"],
            "source_coverage": coverage,
            **evaluation,
        })

    except Exception as e:

        print(
            f"\nPIPELINE ERROR: {e}"
        )

        results_all.append({
            "question": question,
            "answer": "",
            "answerable": item["answerable"],
            "source_coverage": 0,
            "relevance": 0,
            "faithfulness": 0,
            "correctness": 0,
            "hallucination": 0,
            "refusal_correct": 0,
            "reason": str(e),
        })


# ------------------------------------------------------------
# 10. DATAFRAME
# ------------------------------------------------------------

df_results = pd.DataFrame(
    results_all
)


# ------------------------------------------------------------
# 11. SUMMARY
# ------------------------------------------------------------

print("\n\n")
print("=" * 100)
print("GENERATION EVALUATION SUMMARY")
print("=" * 100)

print(
    f"\nQuestions evaluated: "
    f"{len(df_results)}"
)

print(
    f"Answerable questions: "
    f"{df_results['answerable'].sum()}"
)

print(
    f"Unanswerable questions: "
    f"{(~df_results['answerable']).sum()}"
)


avg_relevance = (
    df_results["relevance"].mean()
)

avg_faithfulness = (
    df_results["faithfulness"].mean()
)

avg_correctness = (
    df_results["correctness"].mean()
)

avg_hallucination = (
    df_results["hallucination"].mean()
)

avg_refusal = (
    df_results["refusal_correct"].mean()
)

avg_source_coverage = (
    df_results["source_coverage"].mean()
)


overall_generation_score = (
    avg_relevance
    + avg_faithfulness
    + avg_correctness
    + avg_hallucination
    + avg_refusal
) / 5


print("\nAVERAGE METRICS")
print("-" * 60)

print(
    f"Answer Relevance       : "
    f"{avg_relevance:.2f}/10"
)

print(
    f"Faithfulness           : "
    f"{avg_faithfulness:.2f}/10"
)

print(
    f"Answer Correctness     : "
    f"{avg_correctness:.2f}/10"
)

print(
    f"Hallucination Safety   : "
    f"{avg_hallucination:.2f}/10"
)

print(
    f"Refusal Correctness    : "
    f"{avg_refusal:.2f}/10"
)

print(
    f"Source Coverage        : "
    f"{avg_source_coverage * 100:.1f}%"
)

print(
    f"\nOVERALL GENERATION SCORE: "
    f"{overall_generation_score:.2f}/10"
)


# ------------------------------------------------------------
# 12. ANSWERABLE / UNANSWERABLE
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("ANSWERABLE vs UNANSWERABLE")
print("=" * 100)

answerable_df = df_results[
    df_results["answerable"] == True
]

unanswerable_df = df_results[
    df_results["answerable"] == False
]

if len(answerable_df):

    print("\nANSWERABLE QUESTIONS")

    print(
        f"Relevance    : "
        f"{answerable_df['relevance'].mean():.2f}/10"
    )

    print(
        f"Faithfulness : "
        f"{answerable_df['faithfulness'].mean():.2f}/10"
    )

    print(
        f"Correctness  : "
        f"{answerable_df['correctness'].mean():.2f}/10"
    )


if len(unanswerable_df):

    print("\nUNANSWERABLE QUESTIONS")

    print(
        f"Refusal Correctness : "
        f"{unanswerable_df['refusal_correct'].mean():.2f}/10"
    )

    print(
        f"Hallucination Safety : "
        f"{unanswerable_df['hallucination'].mean():.2f}/10"
    )


# ------------------------------------------------------------
# 13. FAILURE DETECTION
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("GENERATION PROBLEMS")
print("=" * 100)

problems = []

for _, row in df_results.iterrows():

    issues = []

    if row["relevance"] < 7:
        issues.append(
            "low relevance"
        )

    if row["faithfulness"] < 7:
        issues.append(
            "weak grounding"
        )

    if row["correctness"] < 7:
        issues.append(
            "low correctness"
        )

    if row["hallucination"] < 7:
        issues.append(
            "possible hallucination"
        )

    if (
        row["answerable"] == False
        and row["refusal_correct"] < 7
    ):
        issues.append(
            "failed refusal"
        )

    if issues:

        problems.append({
            "question": row["question"],
            "issues": ", ".join(issues),
        })


if problems:

    for problem in problems:

        print(
            f"\nQuestion: "
            f"{problem['question']}"
        )

        print(
            f"Issues: "
            f"{problem['issues']}"
        )

else:

    print(
        "\nNo major generation problems detected."
    )


# ------------------------------------------------------------
# 14. FINAL DECISION
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("FINAL GENERATION DECISION")
print("=" * 100)

if (
    avg_faithfulness >= 8
    and avg_correctness >= 8
    and avg_relevance >= 8
    and avg_hallucination >= 8
    and avg_refusal >= 8
):

    print(
        "\n✅ GENERATION READY"
    )

elif (
    avg_faithfulness >= 7
    and avg_correctness >= 7
    and avg_relevance >= 7
):

    print(
        "\n⚠️ GENERATION ACCEPTABLE"
    )

else:

    print(
        "\n❌ GENERATION NOT READY"
    )


# ------------------------------------------------------------
# 15. COMPACT TABLE
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("QUESTION-LEVEL METRICS")
print("=" * 100)

display_columns = [
    "question",
    "relevance",
    "faithfulness",
    "correctness",
    "hallucination",
    "refusal_correct",
    "source_coverage",
]

display(
    df_results[
        display_columns
    ].round(2)
)

GENERATION EVALUATION

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9753.97it/s]


Loading Chroma vector store...
Initializing hybrid retriever...
Initializing Groq LLM...

Initialization complete.
LLM       : openai/gpt-oss-20b
Embeddings: BAAI/bge-small-en-v1.5
Retriever : Dense + BM25 + RRF


[1/14] What is the software development lifecycle?

ANSWER:
FinSolve’s software development lifecycle (SDLC) is a Scrum‑based Agile process that runs in 2‑week sprints. It covers the full set of activities outlined in the scope of the engineering document: system architecture and infrastructure, technology stack, security and compliance, testing and quality assurance, deployment and DevOps practices, monitoring and maintenance, and the future technology roadmap. (Sources: engineering_master_doc.md – 4.1 Agile Methodology; engineering_master_doc.md – 1.3 Scope)

RETRIEVED SOURCES:
  1. engineering_master_doc.md
  2. engineering_master_doc.md
  3. engineering_master_doc.md
  4. engineering_master_doc.md
  5. employee_handbook.md

GENERATION METRICS:
  Relevance       : 9.0/10
 

,question,relevance,faithfulness,correctness,hallucination,refusal_correct,source_coverage
0,What is the software development lifecycle?,9.0,10.0,10.0,10.0,10.0,1.0
1,What is the company's revenue?,5.0,10.0,10.0,10.0,10.0,1.0
2,What does the marketing department do?,10.0,10.0,10.0,10.0,10.0,1.0
3,How many employees work in the company?,10.0,10.0,10.0,10.0,10.0,0.0
4,What is the average employee salary?,0.0,10.0,10.0,10.0,10.0,1.0
5,What does the engineering team handle?,9.0,9.0,9.0,10.0,10.0,1.0
6,What is the employee referral reward?,10.0,10.0,10.0,10.0,10.0,1.0
7,What were the company's Q1 marketing results?,10.0,10.0,10.0,10.0,10.0,1.0
8,What were the company's Q2 marketing results?,10.0,10.0,10.0,10.0,10.0,1.0
9,What were the company's Q3 marketing results?,0.0,0.0,0.0,10.0,10.0,1.0


## 7. Conversational RAG evaluation

A single-turn test does not show whether the chatbot understands follow-up questions. These cases test whether conversation history supports query rewriting, intent routing, evidence retrieval, and an appropriate answer. The LLM-based faithfulness check is an estimate, so review low-scoring examples manually.

In [15]:
import importlib
import sys
from pathlib import Path

conv_project_root = Path.cwd().parent
if not (conv_project_root / "app").is_dir():
    conv_project_root = Path.cwd()

conv_app_dir = conv_project_root / "app"
if str(conv_app_dir) not in sys.path:
    sys.path.insert(0, str(conv_app_dir))

import access_control as conversational_access_control
conversational_access_control = importlib.reload(conversational_access_control)

import retriever as conversational_retriever
conversational_retriever = importlib.reload(conversational_retriever)

import rag as conversational_rag
conversational_rag = importlib.reload(conversational_rag)

assert conversational_rag.MAX_HISTORY_TURNS == 25
assert conversational_rag.MAX_HISTORY_CHARS == 6000
print(f"App loaded from: {conv_app_dir}")
print(f"History window: {conversational_rag.MAX_HISTORY_TURNS} turns")
print(f"RBAC-aware API: {conversational_rag.answer_question.__defaults__}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9103.73it/s]


App loaded from: /home/khushi/Codebasics/Dataset/DS-RPC-01/app
History window: 25 turns
RBAC-aware API: ('employee', None)


In [16]:
conversational_cases = [
    {
        "case": "finance_quarter_follow_up",
        "role": "finance",
        "seed_question": "What was Q4 revenue?",
        "follow_up": "What about Q3?",
        "standalone_terms": ["q3", "revenue"],
        "expected_intent": "finance_information",
        "expected_source": "quarterly_financial_report.md",
        "answer_terms": ["2.4", "billion"],
        "answerable": True,
        "route_type": "document",
    },
    {
        "case": "structured_hr_pronoun",
        "role": "hr",
        "seed_question": "How many employees are there?",
        "follow_up": "What's their average salary?",
        "standalone_terms": ["employee", "average salary"],
        "expected_intent": "salary_information",
        "expected_source": "hr_data.csv",
        "answer_terms": ["1,187,565.31"],
        "answerable": True,
        "route_type": "structured_hr",
    },
    {
        "case": "marketing_reference_follow_up",
        "role": "marketing",
        "seed_question": "Tell me about Q4 marketing.",
        "follow_up": "What were its main targets?",
        "standalone_terms": ["q4", "marketing", "targets"],
        "expected_intent": "marketing_information",
        "expected_source": "market_report_q4_2024.md",
        "answer_terms": ["220,000", "$11 million", "$2.5 million"],
        "answerable": True,
        "route_type": "document",
    },
    {
        "case": "standalone_topic_switch",
        "role": "employee",
        "seed_question": "How do I apply for leave?",
        "follow_up": "What is the employee referral reward?",
        "standalone_terms": ["employee referral", "reward"],
        "expected_intent": "hr_information",
        "expected_source": "employee_handbook.md",
        "answer_terms": ["10,000"],
        "answerable": True,
        "route_type": "document",
    },
    {
        "case": "unanswerable_follow_up",
        "role": "marketing",
        "seed_question": "What were Q4 marketing targets?",
        "follow_up": "What is the CEO's phone number?",
        "standalone_terms": ["ceo", "phone number"],
        "expected_intent": "unknown",
        "expected_source": None,
        "answer_terms": [],
        "answerable": False,
        "route_type": "unanswerable",
    },
]

print(f"Defined {len(conversational_cases)} conversation tests.")
for case in conversational_cases:
    print(f"- {case['case']} ({case['role']}): {case['seed_question']} -> {case['follow_up']}")

Defined 5 conversation tests.
- finance_quarter_follow_up (finance): What was Q4 revenue? -> What about Q3?
- structured_hr_pronoun (hr): How many employees are there? -> What's their average salary?
- marketing_reference_follow_up (marketing): Tell me about Q4 marketing. -> What were its main targets?
- standalone_topic_switch (employee): How do I apply for leave? -> What is the employee referral reward?
- unanswerable_follow_up (marketing): What were Q4 marketing targets? -> What is the CEO's phone number?


### 7.1 Run the conversations

Each test starts with an empty memory, asks the seed question through the real application, then sends the follow-up. This is important: manually inserting the expected answer would test rewriting, but would not test the actual conversation path. The notebook stores only aggregate HR output, never individual employee records.


In [17]:
conversation_runs = []

for case in conversational_cases:
    conversational_rag.clear_conversation_history()
    seed_result = conversational_rag.answer_question(
        case["seed_question"],
        role=case["role"],
    )
    follow_up_result = conversational_rag.answer_question(
        case["follow_up"],
        role=case["role"],
    )

    conversation_runs.append({
        **case,
        "seed_answer": seed_result["answer"],
        "standalone_question": follow_up_result["standalone_question"],
        "intent": follow_up_result["intent"],
        "search_query": follow_up_result["search_query"],
        "answer": follow_up_result["answer"],
        "sources": sorted({
            result["metadata"].get("source", "unknown")
            for result in follow_up_result["results"]
        }),
        "documents": follow_up_result["results"],
    })

    print(f"Completed: {case['case']} ({case['role']})")

conversational_rag.clear_conversation_history()
print(f"\nCompleted {len(conversation_runs)} conversation tests.")

Completed: finance_quarter_follow_up (finance)
Completed: structured_hr_pronoun (hr)
Completed: marketing_reference_follow_up (marketing)
Completed: standalone_topic_switch (employee)
Completed: unanswerable_follow_up (marketing)

Completed 5 conversation tests.


### 7.2 Deterministic checks

These checks are intentionally simple and repeatable. Rewrite accuracy here means all expected meaning-terms appear in the standalone rewrite; it is not a full linguistic similarity score. Retrieval is checked against the final evidence passed to generation. For HR, the expected route is the structured CSV aggregation, so we check the HR intent instead of expecting CSV rows in Chroma results.


In [18]:
import re
import unicodedata
import pandas as pd


def normalize_eval_text(text):
    normalized = unicodedata.normalize("NFKC", text).casefold()
    normalized = normalized.replace(",", "")
    return re.sub(r"\s+", " ", normalized)


def contains_all_terms(text, terms):
    normalized_text = normalize_eval_text(text)
    return all(normalize_eval_text(term) in normalized_text for term in terms)


def is_safe_refusal(text):
    normalized_text = normalize_eval_text(text)
    refusal_signals = (
        "do not contain enough information",
        "don't contain enough information",
        "cannot provide",
        "can't provide",
        "not available",
        "does not provide",
        "doesn't provide",
        "not found in",
    )
    return any(signal in normalized_text for signal in refusal_signals)


scored_conversations = []
for run in conversation_runs:
    rewrite_ok = contains_all_terms(
        run["standalone_question"], run["standalone_terms"]
    )
    intent_ok = run["intent"] == run["expected_intent"]

    if run["route_type"] == "document":
        route_ok = run["expected_source"] in run["sources"]
        retrieval_ok = route_ok
    elif run["route_type"] == "structured_hr":
        route_ok = intent_ok
        retrieval_ok = None
    else:
        route_ok = True
        retrieval_ok = None

    if run["answerable"]:
        answer_ok = contains_all_terms(run["answer"], run["answer_terms"])
        safe_ok = None
    else:
        answer_ok = None
        safe_ok = is_safe_refusal(run["answer"])

    scored_conversations.append({
        "case": run["case"],
        "follow_up": run["follow_up"],
        "standalone_question": run["standalone_question"],
        "intent": run["intent"],
        "search_query": run["search_query"],
        "sources": ", ".join(run["sources"]),
        "rewrite_correct": rewrite_ok,
        "intent_correct": intent_ok,
        "route_correct": route_ok,
        "retrieval_correct": retrieval_ok,
        "answer_correct": answer_ok,
        "safe_abstention_correct": safe_ok,
        "answer": run["answer"],
    })

conversation_scores_df = pd.DataFrame(scored_conversations)
display(conversation_scores_df.drop(columns=["answer"]).style)

print("\nDeterministic metric summary")
for metric in ("rewrite_correct", "intent_correct", "route_correct", "answer_correct"):
    metric_scores = conversation_scores_df[metric].dropna()
    print(f"{metric}: {metric_scores.mean():.0%} ({len(metric_scores)} cases)")

retrieval_scores = conversation_scores_df["retrieval_correct"].dropna()
if not retrieval_scores.empty:
    print(f"document retrieval correctness: {retrieval_scores.mean():.0%} ({len(retrieval_scores)} cases)")

refusal_scores = conversation_scores_df["safe_abstention_correct"].dropna()
if not refusal_scores.empty:
    print(f"unanswerable refusal correctness: {refusal_scores.mean():.0%} ({len(refusal_scores)} cases)")

,case,follow_up,standalone_question,intent,search_query,sources,rewrite_correct,intent_correct,route_correct,retrieval_correct,answer_correct,safe_abstention_correct
0,finance_quarter_follow_up,What about Q3?,What was Q3 revenue?,finance_information,Q3 2024 quarterly financial revenue,"financial_summary.md, quarterly_financial_report.md",True,True,True,True,True,None
1,structured_hr_pronoun,What's their average salary?,What is the average salary of employees at FinSolve Technologies?,salary_information,,,True,True,True,None,True,None
2,marketing_reference_follow_up,What were its main targets?,What were the main marketing targets for Q4 2024?,marketing_information,Q4 2024 marketing targets conversion rate 15% retention 85% CPA $11.36,"market_report_q4_2024.md, marketing_report_q1_2024.md, marketing_report_q2_2024.md, marketing_report_q3_2024.md",True,True,True,True,True,None
3,standalone_topic_switch,What is the employee referral reward?,What is the employee referral reward?,hr_information,FinSolve employee referral reward policy,employee_handbook.md,True,True,True,True,True,None
4,unanswerable_follow_up,What is the CEO's phone number?,What is the CEO's phone number?,unknown,What is the CEO's phone number?,"employee_handbook.md, market_report_q4_2024.md, marketing_report_2024.md, marketing_report_q1_2024.md",True,True,True,None,None,True



Deterministic metric summary
rewrite_correct: 100% (5 cases)
intent_correct: 100% (5 cases)
route_correct: 100% (5 cases)
answer_correct: 100% (4 cases)
document retrieval correctness: 100% (3 cases)
unanswerable refusal correctness: 100% (1 cases)


### 7.3 Faithfulness and hallucination-safety estimate

A correct-looking answer can still contain extra unsupported claims. For document-backed cases, we therefore ask a separate LLM judge to compare the answer with the exact evidence chunks sent to generation. The judge returns 0–10 faithfulness and hallucination-safety estimates. This is useful for screening, but it is not an objective proof; inspect low-scoring examples manually. The HR average-salary case is validated against the deterministic CSV aggregate and is excluded from this document-context judge.

In [19]:
import json

from generator import build_context


conversation_judge_prompt = """
You are a strict evaluator of an internal company RAG answer.
Judge only against the supplied evidence. The conversation history is
not evidence. Do not reward claims just because they sound plausible.

Question: {question}
Expected answerable: {answerable}
Evidence context:
{context}

Assistant answer:
{answer}

Return only JSON with integer scores from 0 to 10:
{{
  "faithfulness": 0,
  "hallucination_safety": 0,
  "reason": "brief evidence-based reason"
}}
Faithfulness measures whether factual claims are supported by the context.
Hallucination safety is 10 when there are no unsupported factual claims.
For unanswerable questions, a clear abstention is safe; inventing an answer is unsafe.
"""

judge_results = []
for run in conversation_runs:
    if run["route_type"] == "structured_hr":
        continue

    context = build_context(run["documents"])
    prompt = conversation_judge_prompt.format(
        question=run["standalone_question"],
        answerable=run["answerable"],
        context=context,
        answer=run["answer"],
    )

    try:
        response = conversational_rag.llm.invoke(prompt)
        raw = re.sub(r"```json|```", "", response.content, flags=re.IGNORECASE).strip()
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if not match:
            raise ValueError("Judge response did not contain JSON")
        scores = json.loads(match.group())
        judge_results.append({
            "case": run["case"],
            "faithfulness": float(scores["faithfulness"]),
            "hallucination_safety": float(scores["hallucination_safety"]),
            "judge_reason": scores.get("reason", ""),
        })
    except Exception as error:
        judge_results.append({
            "case": run["case"],
            "faithfulness": None,
            "hallucination_safety": None,
            "judge_reason": f"Judge unavailable: {type(error).__name__}",
        })

judge_scores_df = pd.DataFrame(judge_results)
display(judge_scores_df)

valid_judgements = judge_scores_df.dropna(
    subset=["faithfulness", "hallucination_safety"]
)
if not valid_judgements.empty:
    print("\nMean LLM-judge estimates (0-10)")
    print(f"Faithfulness: {valid_judgements['faithfulness'].mean():.2f}/10")
    print(f"Hallucination safety: {valid_judgements['hallucination_safety'].mean():.2f}/10")
else:
    print("No judge scores available; use deterministic results above.")

,case,faithfulness,hallucination_safety,judge_reason
0,finance_quarter_follow_up,10.0,10.0,The assistant correctly cites the Q3 revenue f...
1,marketing_reference_follow_up,10.0,10.0,All claimed targets are directly cited from So...
2,standalone_topic_switch,10.0,10.0,"The answer correctly states the ₹10,000 reward..."
3,unanswerable_follow_up,10.0,10.0,"The assistant correctly abstains, stating that..."



Mean LLM-judge estimates (0-10)
Faithfulness: 10.00/10
Hallucination safety: 10.00/10


### Results from this run

On this five-dialogue smoke benchmark: rewrite accuracy was **5/5**, intent accuracy **5/5**, route accuracy **5/5**, document retrieval correctness **3/3**, answer-fact accuracy **4/4**, and safe refusal **1/1**. The structured HR case was verified through the CSV aggregate route, not document retrieval. On four document-context cases, the LLM judge estimated faithfulness and hallucination safety at **10/10** each.

## 8. Department-level access control

Authentication identifies the caller; authorization determines which departments that caller may access. The tests below check role normalization and department allow-lists, helping ensure restricted documents do not reach generation.

### Access-control implementation notes

RAG access control has two parts: **authorization** decides which departments a role may query, and **retrieval filtering** makes sure documents outside that allow-list never reach generation. This project applies the same allow-list to dense Chroma search and BM25 candidates, then checks the fused results again.

| Role | Allowed document departments | Structured HR aggregates |
|---|---|---|
| Employee | General | No |
| Finance | General, Finance | No |
| Marketing | General, Marketing | No |
| HR | General, HR | Yes |
| Engineering | General, Engineering | No |
| Executive / C-Level | All current departments | Yes |



In [20]:
from unittest.mock import patch

from access_control import get_allowed_departments, normalize_role

expected_access = {
    "employee": {"general"},
    "finance": {"finance", "general"},
    "marketing": {"marketing", "general"},
    "hr": {"hr", "general"},
    "engineering": {"engineering", "general"},
    "executive": {"engineering", "finance", "general", "hr", "marketing"},
}

for role, expected_departments in expected_access.items():
    assert get_allowed_departments(role) == expected_departments
assert normalize_role("C-Level") == "executive"

rbac_retrieval_checks = []
for role, allowed_departments in expected_access.items():
    results = conversational_rag.retriever.retrieve(
        "Q3 2024 quarterly financial revenue",
        k=10,
        departments=allowed_departments,
    )
    result_departments = {
        result["metadata"].get("department")
        for result in results
    }
    assert result_departments <= allowed_departments
    rbac_retrieval_checks.append({
        "role": role,
        "allowed_departments": ", ".join(sorted(allowed_departments)),
        "returned_departments": ", ".join(sorted(result_departments)),
        "returned_chunks": len(results),
    })

def canned_route(intent):
    return lambda question, history: {
        "intent": intent,
        "standalone_question": question,
        "search_query": question,
    }

conversational_rag.clear_conversation_history()
with patch.object(
    conversational_rag,
    "understand_query",
    side_effect=canned_route("finance_information"),
):
    with patch.object(
        conversational_rag.retriever,
        "retrieve",
        side_effect=AssertionError("Unauthorized request reached retrieval"),
    ):
        employee_finance = conversational_rag.answer_question(
            "Q3 revenue",
            role="employee",
        )
assert employee_finance["access_denied"] is True
assert employee_finance["results"] == []

conversational_rag.clear_conversation_history()
with patch.object(
    conversational_rag,
    "understand_query",
    side_effect=canned_route("salary_information"),
):
    with patch.object(
        conversational_rag,
        "_structured_hr_answer",
        side_effect=AssertionError("Unauthorized request reached HR data"),
    ):
        employee_salary = conversational_rag.answer_question(
            "average salary",
            role="employee",
        )
assert employee_salary["access_denied"] is True

captured_allowed_departments = []
conversational_rag.clear_conversation_history()
with patch.object(
    conversational_rag,
    "understand_query",
    side_effect=canned_route("finance_information"),
):
    with patch.object(
        conversational_rag.retriever,
        "retrieve",
        side_effect=lambda query, k, departments: (
            captured_allowed_departments.append(set(departments)) or []
        ),
    ):
        with patch.object(
            conversational_rag,
            "generate_answer",
            return_value="test answer",
        ):
            finance_result = conversational_rag.answer_question(
                "Q3 revenue",
                role="finance",
            )
assert finance_result["access_denied"] is False
assert captured_allowed_departments == [{"finance", "general"}]
conversational_rag.clear_conversation_history()

display(pd.DataFrame(rbac_retrieval_checks))
print("PASS: role matrix, filtered retrieval, deny-before-retrieval, deny-before-HR, and authorized Finance access.")

,role,allowed_departments,returned_departments,returned_chunks
0,employee,general,general,10
1,finance,"finance, general",finance,10
2,marketing,"general, marketing",marketing,10
3,hr,"general, hr","general, hr",10
4,engineering,"engineering, general","engineering, general",10
5,executive,"engineering, finance, general, hr, marketing","finance, marketing",10


PASS: role matrix, filtered retrieval, deny-before-retrieval, deny-before-HR, and authorized Finance access.


## 9. FastAPI backend checks

FastAPI exposes the chatbot through HTTP. The existing automated tests check the API's basic behavior, trusted roles, access denial, and session isolation. The API implementation belongs in `app/api.py`; this notebook only verifies it.

### Production architecture overview

A **production architecture** is the set of pieces that let an app serve real people safely and predictably. We are adding an HTTP front door to the RAG pipeline, but keeping the existing retrieval and grounded-answer logic intact.




### Run the API tests

The next cell runs the project's automated tests once.

In [21]:
import os
import subprocess
import sys

project_root = Path.cwd().parent if Path.cwd().name == "evaluation" else Path.cwd()
environment = os.environ.copy()
app_path = str(project_root / "app")
existing_path = environment.get("PYTHONPATH", "")
environment["PYTHONPATH"] = os.pathsep.join(
    item for item in (app_path, existing_path) if item
)

test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(project_root / "tests"),
        "-v",
    ],
    cwd=project_root,
    env=environment,
    capture_output=True,
    text=True,
)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


test_health_does_not_require_rag_or_authentication (test_api.ChatApiTests.test_health_does_not_require_rag_or_authentication) ... ok
test_missing_or_invalid_credentials_do_not_call_rag (test_api.ChatApiTests.test_missing_or_invalid_credentials_do_not_call_rag) ... ok
test_rag_access_denial_becomes_http_403 (test_api.ChatApiTests.test_rag_access_denial_becomes_http_403) ... ok
test_server_identity_controls_role_and_memory_is_isolated (test_api.ChatApiTests.test_server_identity_controls_role_and_memory_is_isolated) ... ok
test_preserves_fastapi_authorization_denial (test_frontend_client.FrontendClientTests.test_preserves_fastapi_authorization_denial) ... ok
test_sends_bearer_token_session_and_question (test_frontend_client.FrontendClientTests.test_sends_bearer_token_session_and_question) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.028s

OK



## 10. Streamlit frontend

Streamlit is the chat UI; it sends user questions to FastAPI. Its implementation belongs in the frontend file, not duplicated in this evaluation notebook. Start FastAPI first, then run the UI from the project root with `streamlit run frontend/app.py`.

The Streamlit UI is checked manually through the running browser interface; the same unit-test suite is not rerun here. The end-to-end evaluation below checks the RAG application logic.

## 11. End-to-end evaluation

Run five representative tests through the actual RAG application: permitted Finance access, denied Employee access, structured HR aggregation, safe refusal, and session isolation. This checks the main components together.

In [26]:
# ==============================================================================
# STEP 2: ROBUST END-TO-END RAG EVALUATION (ALL 5 CHECKS)
# ==============================================================================

import re
import sys
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

from groq import RateLimitError

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "evaluation"
    else Path.cwd()
)

APP_DIR = PROJECT_ROOT / "app"

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

import rag
import conversation_store as cs_module


# ------------------------------------------------------------------------------
# CONVERSATION STORE INITIALIZATION
# ------------------------------------------------------------------------------

if hasattr(cs_module, "conversation_store"):
    conv_store = cs_module.conversation_store
elif hasattr(cs_module, "ConversationStore"):
    conv_store = cs_module.ConversationStore()
else:
    conv_store = cs_module


# ------------------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------------------

def clean_text(text: str) -> str:
    """Normalize text for reliable, case-insensitive comparisons."""
    norm = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", norm).strip()


def get_retry_message(exc: RateLimitError) -> str:
    """
    Return a polite retry message without displaying raw API details.

    Uses the server's Retry-After header when available.
    Falls back to a generic message if the retry time is unavailable.
    """
    response = getattr(exc, "response", None)
    retry_after = None

    if response is not None:
        retry_after = response.headers.get("retry-after")

    if retry_after:
        try:
            # Retry-After can be a number of seconds.
            seconds = max(0, int(float(retry_after)))
            minutes, remaining_seconds = divmod(seconds, 60)

            if minutes:
                wait_time = f"{minutes} minute(s) and {remaining_seconds} second(s)"
            else:
                wait_time = f"{remaining_seconds} second(s)"

            return (
                "The AI service has temporarily reached its usage limit. "
                f"Please retry in approximately {wait_time}."
            )

        except (ValueError, TypeError):
            try:
                # Retry-After can also be an HTTP date.
                retry_time = parsedate_to_datetime(retry_after)

                if retry_time.tzinfo is None:
                    retry_time = retry_time.replace(tzinfo=timezone.utc)

                seconds = max(
                    0,
                    int(
                        (
                            retry_time
                            - datetime.now(timezone.utc)
                        ).total_seconds()
                    ),
                )

                minutes, remaining_seconds = divmod(seconds, 60)

                if minutes:
                    wait_time = (
                        f"{minutes} minute(s) and "
                        f"{remaining_seconds} second(s)"
                    )
                else:
                    wait_time = f"{remaining_seconds} second(s)"

                return (
                    "The AI service has temporarily reached its usage limit. "
                    f"Please retry in approximately {wait_time}."
                )

            except (ValueError, TypeError, OverflowError):
                pass

    return (
        "The AI service has temporarily reached its usage limit. "
        "Please wait a few minutes before retrying."
    )


def safe_answer(question: str, role: str):
    """
    Execute one RAG request safely.

    Returns:
        dict: Successful RAG response.
        None: Request was blocked by an API rate limit.
    """
    try:
        return rag.answer_question(question, role=role)

    except RateLimitError as exc:
        print(f"\n  {get_retry_message(exc)}")
        return None


def show_verdict(result, passed: bool) -> str:
    """Distinguish successful tests, failed assertions, and blocked tests."""
    if result is None:
        return "BLOCKED — API rate limit"

    return "PASS" if passed else "FAIL"


# ------------------------------------------------------------------------------
# START EVALUATION
# ------------------------------------------------------------------------------

print("=" * 80)
print("RUNNING FINAL END-TO-END RAG EVALUATION")
print("=" * 80)

# Track test status independently so blocked tests are not counted as failures.
results_summary = {}


# ------------------------------------------------------------------------------
# CASE 1: Permitted Request (Finance -> Q3 Revenue)
# ------------------------------------------------------------------------------

rag.clear_conversation_history()

res1 = safe_answer(
    "What was the company revenue in Q3 2024?",
    role="finance",
)

ans1_norm = clean_text(res1.get("answer", "")) if res1 else ""

c1_ok = (
    res1 is not None
    and not res1.get("access_denied", False)
    and "2.4" in ans1_norm
    and "billion" in ans1_norm
)

print("\n[Case 1] Finance User -> Q3 Revenue:")

if res1 is not None:
    print(
        f"  Access Denied : {res1.get('access_denied', False)} "
        "(Expected: False)"
    )
    print(f"  Answer        : {res1.get('answer', '').strip()}")

print(f"  Verdict       : {show_verdict(res1, c1_ok)}")
results_summary["Case 1"] = show_verdict(res1, c1_ok)


# ------------------------------------------------------------------------------
# CASE 2: Denied Request (Employee -> Finance Data - RBAC Barrier)
# ------------------------------------------------------------------------------

rag.clear_conversation_history()

res2 = safe_answer(
    "What was the company revenue in Q3 2024?",
    role="employee",
)

c2_ok = (
    res2 is not None
    and res2.get("access_denied", False)
    and len(res2.get("results", [])) == 0
)

print("\n[Case 2] Employee User -> Q3 Revenue (RBAC Barrier):")

if res2 is not None:
    print(
        f"  Access Denied : {res2.get('access_denied', False)} "
        "(Expected: True)"
    )
    print(
        f"  Docs Retrieved: {len(res2.get('results', []))} "
        "(Expected: 0)"
    )

print(f"  Verdict       : {show_verdict(res2, c2_ok)}")
results_summary["Case 2"] = show_verdict(res2, c2_ok)


# ------------------------------------------------------------------------------
# CASE 3: Structured HR Aggregation (HR User)
# ------------------------------------------------------------------------------

rag.clear_conversation_history()

res3 = safe_answer(
    "How many employees are there in the company?",
    role="hr",
)

ans3_norm = clean_text(res3.get("answer", "")) if res3 else ""

c3_ok = (
    res3 is not None
    and not res3.get("access_denied", False)
    and "100" in ans3_norm
)

print("\n[Case 3] HR User -> Employee Count (Structured HR):")

if res3 is not None:
    print(f"  Intent        : {res3.get('intent')}")
    print(f"  Answer        : {res3.get('answer', '').strip()}")

print(f"  Verdict       : {show_verdict(res3, c3_ok)}")
results_summary["Case 3"] = show_verdict(res3, c3_ok)


# ------------------------------------------------------------------------------
# CASE 4: Unanswerable Question (Safe Abstention / Refusal)
# ------------------------------------------------------------------------------

rag.clear_conversation_history()

res4 = safe_answer(
    "What is the company stock price today?",
    role="finance",
)

ans4_norm = clean_text(res4.get("answer", "")) if res4 else ""

refusal_phrases = [
    "not contain enough information",
    "not mentioned",
    "unavailable",
    "cannot provide",
]

c4_ok = (
    res4 is not None
    and any(phrase in ans4_norm for phrase in refusal_phrases)
)

print("\n[Case 4] Finance User -> Stock Price (Safe Abstention):")

if res4 is not None:
    print(f"  Answer        : {res4.get('answer', '').strip()}")

print(f"  Verdict       : {show_verdict(res4, c4_ok)}")
results_summary["Case 4"] = show_verdict(res4, c4_ok)


# ------------------------------------------------------------------------------
# CASE 5: Session Isolation (In-Memory Store)
# ------------------------------------------------------------------------------

session_a = "11111111-1111-1111-1111-111111111111"
session_b = "22222222-2222-2222-2222-222222222222"

history_b = []
history_a_as_bob = []

try:
    if hasattr(conv_store, "clear"):
        conv_store.clear()
    elif hasattr(conv_store, "reset"):
        conv_store.reset()

    if (
        hasattr(conv_store, "append_message")
        and hasattr(conv_store, "get_history")
    ):
        conv_store.append_message(
            "alice",
            session_a,
            "user",
            "My secret finance project",
        )

        conv_store.append_message(
            "alice",
            session_a,
            "assistant",
            "Secret project acknowledged",
        )

        history_b = conv_store.get_history("bob", session_b)
        history_a_as_bob = conv_store.get_history("bob", session_a)

        c5_ok = (
            len(history_b) == 0
            and len(history_a_as_bob) == 0
        )

        c5_status = "PASS" if c5_ok else "FAIL"

    else:
        c5_ok = False
        c5_status = "BLOCKED — required store methods unavailable"

except Exception:
    # Do not expose internal store errors or incorrectly report a pass.
    c5_ok = False
    c5_status = "BLOCKED — session isolation test could not execute"

print("\n[Case 5] Multi-user Session Isolation:")
print(
    f"  Bob reading session B messages    : {len(history_b)} "
    "(Expected: 0)"
)
print(
    f"  Bob trying to read Alice session A: {len(history_a_as_bob)} "
    "(Expected: 0)"
)
print(f"  Verdict                           : {c5_status}")

results_summary["Case 5"] = c5_status


# ------------------------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL EVALUATION SUMMARY")
print("=" * 80)

for case, status in results_summary.items():
    print(f"{case}: {status}")

passed_count = sum(
    status == "PASS"
    for status in results_summary.values()
)

failed_count = sum(
    status == "FAIL"
    for status in results_summary.values()
)

blocked_count = sum(
    status.startswith("BLOCKED")
    for status in results_summary.values()
)

print("-" * 80)
print(f"Passed  : {passed_count}/5")
print(f"Failed  : {failed_count}/5")
print(f"Blocked : {blocked_count}/5")

if passed_count == 5:
    print("FINAL VERDICT: ALL 5 CHECKS PASSED")
elif blocked_count > 0:
    print("FINAL VERDICT: INCOMPLETE — RETRY BLOCKED CHECKS")
else:
    print("FINAL VERDICT: SOME CHECKS FAILED")

print("=" * 80)

RUNNING FINAL END-TO-END RAG EVALUATION

[Case 1] Finance User -> Q3 Revenue:
  Access Denied : False (Expected: False)
  Answer        : The company’s revenue in Q3 2024 was **$2.4 billion** (July – September 2024).  

**Source:** quarterly_financial_report.md – “Q3 - July to September 2024 | Quarterly Financial Overview.”
  Verdict       : PASS

[Case 2] Employee User -> Q3 Revenue (RBAC Barrier):
  Access Denied : True (Expected: True)
  Docs Retrieved: 0 (Expected: 0)
  Verdict       : PASS

[Case 3] HR User -> Employee Count (Structured HR):
  Intent        : employee_count
  Answer        : FinSolve Technologies has 100 employees.
  Verdict       : PASS

  The AI service has temporarily reached its usage limit. Please retry in approximately 2 minute(s) and 14 second(s).

[Case 4] Finance User -> Stock Price (Safe Abstention):
  Verdict       : BLOCKED — API rate limit

[Case 5] Multi-user Session Isolation:
  Bob reading session B messages    : 0 (Expected: 0)
  Bob trying to rea

# Final notes: project workflow

1. **Ingestion:** Loaded company documents and HR CSV data; split documents into chunks and stored chunk embeddings in ChromaDB.
2. **Dense retrieval baseline:** Tested semantic search and checked whether expected source files appeared in the top-k results.
3. **Retrieval diagnosis:** Investigated missed questions, including marketing reports and employee-referral information.
4. **Hybrid search:** Combined ChromaDB dense retrieval with BM25 keyword search and fused rankings with Reciprocal Rank Fusion (RRF).
5. **Retrieval evaluation:** Compared top-k results and context strategies using source-based metrics such as Recall@K, precision, and MRR where implemented.
6. **Structured HR handling:** Checked CSV-based employee count and average-salary calculations separately from document search.
7. **Generation evaluation:** Ran one shared loop across all 14 test questions; each case goes through retrieval, context building, answer generation, and LLM-judge scoring.
8. **Conversation evaluation:** Tested follow-up questions, query rewriting, routing, and safe responses across multiple turns.
9. **Access control:** Checked role-to-department permissions and verified unauthorized requests are blocked before protected retrieval/data access.
10. **Application checks:** Ran the API test suite, used Streamlit as the frontend, and ran five end-to-end checks across access, HR data, refusal, and session isolation.

**Important distinction:** The 14 questions are a benchmark dataset used to measure the system. The application does not contain separate generation code for each question. Retrieval metrics measure evidence retrieval; generation metrics measure answer quality and grounding. LLM-judge scores are estimates, so review failed cases and expand the test set over time.